# Treinamento do Modelo LSTM para Previsão de Preços de Ações

Este notebook demonstra o processo completo de coleta de dados, pré-processamento, treinamento e avaliação de um modelo LSTM para prever o preço de fechamento de ações. Ao final, o modelo treinado e o scaler serão salvos para uso na API.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import yfinance as yf
import joblib
import os

## 1. Coleta de Dados

In [2]:
# Definir o símbolo da ação e o período
symbol = 'PETR4.SA'  # Exemplo: Petrobras (ajuste conforme necessário)
start_date = '2018-01-01'
end_date = '2024-07-20'
sequence_length = 60 # Comprimento da sequência para o LSTM

# Baixar os dados
df = yf.download(symbol, start=start_date, end=end_date)

print(f"Dados baixados para {symbol} de {start_date} a {end_date}.")
print(df.head())

C:\Users\Gustavo Imbelloni\AppData\Local\Temp\ipykernel_10996\1530814651.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

Dados baixados para PETR4.SA de 2018-01-01 a 2024-07-20.
Price          Close      High       Low      Open    Volume
Ticker      PETR4.SA  PETR4.SA  PETR4.SA  PETR4.SA  PETR4.SA
Date                                                        
2018-01-02  4.410663  4.410663  4.314722  4.314722  33461800
2018-01-03  4.450638  4.455968  4.362691  4.394671  55940900
2018-01-04  4.458633  4.519929  4.429318  4.471959  37064900
2018-01-05  4.485284  4.493280  4.415993  4.450639  26958200
2018-01-08  4.538585  4.538585  4.453303  4.461298  28400000


## 2. Pré-processamento dos Dados

In [4]:
# Filtrar apenas a coluna 'Close'
data = df['Close']
dataset = data.values.reshape(-1, 1)

# Escalonar os dados
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(dataset)

# Dividir os dados em treino e teste (80% treino, 20% teste)
training_data_len = int(np.ceil(len(scaled_data) * .8))

train_data = scaled_data[0:training_data_len, :]
x_train = []
y_train = []

for i in range(sequence_length, len(train_data)):
    x_train.append(train_data[i-sequence_length:i, 0])
    y_train.append(train_data[i, 0])

x_train, y_train = np.array(x_train), np.array(y_train)
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

# Dados de teste
test_data = scaled_data[training_data_len - sequence_length:, :]
x_test = []
y_test = dataset[training_data_len:, :]

for i in range(sequence_length, len(test_data)):
    x_test.append(test_data[i-sequence_length:i, 0])

x_test = np.array(x_test)
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

print(f"X_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (1241, 60, 1)
y_train shape: (1241,)
X_test shape: (325, 60, 1)
y_test shape: (325, 1)


## 3. Construção e Treinamento do Modelo LSTM

In [5]:
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(Dropout(0.2))
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=25))
model.add(Dense(units=1))

model.compile(optimizer='adam', loss='mean_squared_error')

model.summary()

c:\Users\Gustavo Imbelloni\Desktop\modelo_preditivo_de_redes_neurais\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 50)         │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 25)             │         1,275 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,901 (124.61 KB)

 Trainable params: 31,901 (124.61 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Treinar o modelo
epochs = 25
batch_size = 32

history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, verbose=1)

Epoch 1/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 0.0058
Epoch 2/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 9.3167e-04
Epoch 3/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 8.3033e-04
Epoch 4/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 7.3454e-04
Epoch 5/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 6.7281e-04
Epoch 6/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 7.1646e-04
Epoch 7/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 5.3541e-04
Epoch 8/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 5.6857e-04
Epoch 9/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 5.6943e-04
Epoch 10/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 4.7538e-04
Epoch 11/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 4.6998e-04
Epoch 12/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 4.9532e-04
Epoch 13/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 4.3834e-04
Epoch 14/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 4.0158e-04
Epoch 15/25
39/39 ━

## 4. Avaliação do Modelo

In [7]:
# Fazer previsões no conjunto de teste
predictions = model.predict(x_test)

# Inverter a escala das previsões e dos valores reais
predictions = scaler.inverse_transform(predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
mape = np.mean(np.abs((y_test - predictions) / y_test)) * 100

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"MAPE: {mape:.2f}%")

11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step
RMSE: 1.2482204678624857
MAE: 1.0589474457960861
MAPE: 4.06%


In [ ]:
data_df = df[['Close']].copy()

# Plotar os resultados
train = data_df[:training_data_len]
valid = data_df[training_data_len:].copy()
valid['Predictions'] = predictions

plt.figure(figsize=(16,8))
plt.title('Modelo de Previsão - PETR4', fontsize=18)
plt.xlabel('Data', fontsize=14)
plt.ylabel('Preço de Fechamento (R$)', fontsize=14)

plt.plot(train['Close'])
plt.plot(valid['Close'])
plt.plot(valid['Predictions'])

plt.legend(['Treino', 'Validação', 'Previsões'], loc='lower right')
plt.show()

TypeError: 'method' object does not support item assignment

## 5. Salvamento e Exportação do Modelo e Scaler

In [ ]:
# Criar diretório para salvar o modelo e o scaler, se não existir
model_dir = 'model'
os.makedirs(model_dir, exist_ok=True)

# Salvar o modelo
model_path = os.path.join(model_dir, 'lstm_model.h5')
model.save(model_path)
print(f"Modelo salvo em: {model_path}")

# Salvar o scaler
scaler_path = os.path.join(model_dir, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"Scaler salvo em: {scaler_path}")

Após executar este notebook, os arquivos `lstm_model.h5` e `scaler.pkl` estarão disponíveis no diretório `model/`, prontos para serem utilizados pela API.